# Notebook 15 — Book Introduction

Publication-ready tables and figures for the Introduction of *Fragrance Accord Intelligence*.

## Introduction visualisation inventory

| Item | Title | Purpose |
|---|---|---|
| Table I.1 | Dataset Profile | Establish dataset scale and data completeness |
| Figure I.1 | Analytical Framework | Explain the research workflow and analytical structure |
| Figure I.2 | Dataset Availability Matrix | Show the overlap between rating and valid accord data |

Only items intended for the Introduction are included here. Chapter-specific findings are developed in their respective chapter notebooks.

**Interpretation principles**
- Catalogue prevalence describes representation in the source catalogue, not sales or consumer demand.
- Ratings and rating counts are platform observations, not objective quality measures, sales or unique-consumer counts.
- Accord labels are recorded olfactory characterisations, not ingredient or chemical-composition data.
- Findings, interpretations and hypotheses must remain clearly distinguished.

## 1. Setup and source data

Load the cleaned Fragrantica catalogue and reproduce the valid-accord parsing logic used in the analytical notebooks.

In [0]:
# Install the two publication-oriented libraries used in this notebook.
# Run this cell once per Databricks cluster/environment.
%pip install lets-plot great-tables

In [0]:
# ============================================================
# NOTEBOOK 15 — BOOK INTRODUCTION
# Chapter 1: Mapping the Fragrance Landscape
# ============================================================

from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import base64
import io

from great_tables import GT, style, loc
from lets_plot import (
    LetsPlot, ggplot, aes, geom_bar, geom_text, coord_flip,
    labs, theme, theme_minimal, element_text, element_line,
    element_rect, element_blank, scale_fill_manual, scale_y_continuous,
    ggsave
)

# Configure Lets-Plot for notebook rendering.
LetsPlot.setup_html()

# Source table
SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
df_perfumes = spark.table(SOURCE_TABLE)

# Parse main_accords using the same logic as Notebook 08
df_accords = df_perfumes.withColumn(
    "accords",
    F.from_json(
        F.regexp_replace(
            F.regexp_replace(
                F.col("main_accords"),
                "'",
                '"'
            ),
            r"\bNone\b",
            "null"
        ),
        "array<string>"
    )
)

# One row per fragrance–accord combination.
# Null and empty accord labels are excluded, as in Notebook 08.
df_fragrance_accords = (
    df_accords
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.explode("accords").alias("accord")
    )
    .withColumn("accord", F.lower(F.trim(F.col("accord"))))
    .filter(
        F.col("accord").isNotNull()
        & (F.col("accord") != "")
    )
)

print(f"Source table: {SOURCE_TABLE}")
print(f"Source records: {df_perfumes.count():,}")
print(f"Distinct fragrance IDs: {df_perfumes.select('id').distinct().count():,}")
print(f"Distinct valid accord labels: {df_fragrance_accords.select('accord').distinct().count():,}")

## 2. Shared publication design system

Use the established ivory, espresso and bronze visual language and shared typography and sizing helpers.

In [0]:
# ============================================================
# GLOBAL VISUALISATION DESIGN SYSTEM — ALL TABLES AND CHARTS
# ============================================================

# Single source of truth for colours, typography and sizing.
BOOK_STYLE = {
    "background": "#F7F3EC",
    "surface": "#F7F3EC",
    "text": "#302824",
    "muted": "#756B64",
    "accent": "#A47C59",
    "accent_light": "#D8C3AD",
    "rule": "#E5DDD2",
    "grid": "#E9E2D8",
    "men": "#687F8C",
    "women": "#B47C83",
    "unisex": "#A58B68",
    "other": "#B9B2AC"
}

BOOK_TYPOGRAPHY = {
    "display": "DejaVu Serif",
    "body": "DejaVu Sans"
}

BOOK_FIGURE_CHART_TOP_IN = 1.45  # Fixed physical distance from figure top to chart area

# Shared horizontal-bar layout: chart body starts closer to the left edge.
BOOK_HEADER_X = 0.06
BOOK_BAR_LEFT = 0.15

def apply_book_bar_layout(fig, bottom=0.10, right=0.96):
    """Apply the same left plot margin to every horizontal bar chart."""
    fig.subplots_adjust(
        left=BOOK_BAR_LEFT,
        right=right,
        top=1 - BOOK_FIGURE_CHART_TOP_IN / fig.get_figheight(),
        bottom=bottom
    )

BOOK_SIZES = {
    "title": 22,
    "label": 12,                 # shared readable axis/category/value labels in points
    "bar_value_mm": 4.2,         # Lets-Plot geom_text uses millimetres; approximately 12 pt
    "bar_width": 0.30,           # consistent slim bar thickness across all horizontal charts
    "title_pt": 22,                # Shared figure title size in points
    "table_body_px": "13px",
    "table_title_px": "26px",
    "table_subtitle_px": "15px",
    "section_pt": 10,
    "subtitle_pt": 13,              # Readable shared figure subtitle size in points
    "category_pt": 12,              # Shared category and axis-label size in points
    "value_pt": 12,                 # Match category labels and data-label typography
    "footer_pt": 10,
    "matrix_value_pt": 24
}

plt.rcParams.update({
    "figure.facecolor": BOOK_STYLE["background"],
    "axes.facecolor": BOOK_STYLE["background"],
    "savefig.facecolor": BOOK_STYLE["background"],
    "figure.edgecolor": BOOK_STYLE["background"],
    "text.color": BOOK_STYLE["text"],
    "axes.labelcolor": BOOK_STYLE["text"],
    "xtick.color": BOOK_STYLE["text"],
    "ytick.color": BOOK_STYLE["text"],
    "font.family": BOOK_TYPOGRAPHY["body"],
    "font.size": BOOK_SIZES["label"],
    "axes.titlesize": BOOK_SIZES["title"],
    "axes.titleweight": "bold",
    "axes.titlelocation": "center",
    "axes.labelsize": BOOK_SIZES["label"],
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

def make_book_table(data, title, subtitle=None):
    """Build a table using the global ivory, typography and title rules."""
    table = (
        GT(data)
        .tab_header(title=title, subtitle=subtitle)
        .tab_options(
            table_background_color=BOOK_STYLE["background"],
            heading_background_color=BOOK_STYLE["background"],
            column_labels_background_color=BOOK_STYLE["background"],
            table_font_names=[BOOK_TYPOGRAPHY["body"], "Arial", "sans-serif"],
            table_font_size=BOOK_SIZES["table_body_px"],
            table_font_color=BOOK_STYLE["text"],
            heading_align="left",
            heading_title_font_size=BOOK_SIZES["table_title_px"],
            heading_subtitle_font_size=BOOK_SIZES["table_subtitle_px"],
            heading_padding="8px",
            column_labels_font_weight="bold",
            column_labels_padding="9px",
            table_body_hlines_color=BOOK_STYLE["grid"],
            table_body_hlines_width="1px",
            data_row_padding="6.5px"
        )
    )
    table = table.tab_style(
        style.fill(color=BOOK_STYLE["background"]),
        locations=[loc.body(), loc.column_header(), loc.header()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["display"], size=BOOK_SIZES["table_title_px"], weight="bold", color=BOOK_STYLE["text"]),
        locations=[loc.title()]
    )
    table = table.tab_style(
        style.text(font=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["table_subtitle_px"], color=BOOK_STYLE["muted"]),
        locations=[loc.subtitle()]
    )
    return table

def add_book_figure_header(fig, title, subtitle, section="FRAGRANCE INTELLIGENCE  /  CHAPTER 1"):
    """Apply identical header coordinates, typography and physical spacing to every figure."""
    figure_height = fig.get_figheight()
    section_y = 1 - 0.203 / figure_height
    title_y = 1 - 0.435 / figure_height
    subtitle_y = 1 - 0.841 / figure_height

    fig.text(
        BOOK_HEADER_X, section_y, section,
        color=BOOK_STYLE["accent"],
        fontsize=BOOK_SIZES["section_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, title_y, title,
        color=BOOK_STYLE["text"],
        fontsize=BOOK_SIZES["title_pt"],
        fontfamily=BOOK_TYPOGRAPHY["display"],
        fontweight="bold",
        ha="left",
        va="top"
    )
    fig.text(
        BOOK_HEADER_X, subtitle_y, subtitle,
        color=BOOK_STYLE["muted"],
        fontsize=BOOK_SIZES["subtitle_pt"],
        fontfamily=BOOK_TYPOGRAPHY["body"],
        ha="left",
        va="top"
    )

def show_book_table(table):
    html = table.as_raw_html(all_important=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_plot(plot):
    html = plot.to_html(iframe=True)
    displayHTML(f'<div class="nodarkmode">{html}</div>')

def show_book_matplotlib(fig):
    buffer = io.BytesIO()
    # Keep the full figure canvas so titles retain the same visible left position
    # across figures with different tick-label widths.
    fig.savefig(
        buffer, format="png", dpi=240, bbox_inches=None,
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["background"]
    )
    image_b64 = base64.b64encode(buffer.getvalue()).decode("ascii")
    displayHTML(
        '<div class="nodarkmode">'
        f'<img src="data:image/png;base64,{image_b64}" '
        'style="display:block;max-width:100%;height:auto;'
        f'background:{BOOK_STYLE["background"]};">'
        '</div>'
    )

BOOK_PLOT_THEME = (
    theme_minimal()
    + theme(
        plot_title=element_text(
            family=BOOK_TYPOGRAPHY["display"],
            size=BOOK_SIZES["title_pt"],
            face="bold",
            color=BOOK_STYLE["text"],
            hjust=0.5
        ),
        plot_subtitle=element_text(
            family=BOOK_TYPOGRAPHY["body"],
            size=BOOK_SIZES["subtitle_pt"],
            color=BOOK_STYLE["muted"],
            hjust=0.5
        ),
        axis_title=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["muted"]
        ),
        axis_text=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_text_y=element_text(
            family=BOOK_TYPOGRAPHY["body"], size=BOOK_SIZES["label"], color=BOOK_STYLE["text"]
        ),
        axis_ticks=element_blank(),
        panel_grid_major_y=element_blank(),
        panel_grid_major_x=element_line(color=BOOK_STYLE["grid"], size=0.6),
        panel_grid_minor=element_blank(),
        panel_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        plot_background=element_rect(fill=BOOK_STYLE["background"], color=BOOK_STYLE["background"]),
        legend_position="none"
    )
)

print("Global visualisation standards are ready.")

## 3. Table I.1 — Dataset Profile

The profile distinguishes records with a non-null source accord field from records that produce at least one valid parsed accord. These are not interchangeable measures.

In [0]:
# ============================================================
# TABLE I.1 — DATASET PROFILE
# ============================================================

total_records = df_perfumes.count()
distinct_ids = df_perfumes.select("id").distinct().count()

valid_accord_ids = (
    df_fragrance_accords
    .select("id")
    .distinct()
)

rated_ids = (
    df_perfumes
    .filter(F.col("rating_value").isNotNull())
    .select("id")
    .distinct()
)

profile = df_perfumes.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("id").alias("distinct_ids"),
    F.countDistinct(
        F.when(F.col("rating_value").isNotNull(), F.col("id"))
    ).alias("records_with_rating"),
    F.countDistinct(
        F.when(F.col("rating_count").isNotNull(), F.col("id"))
    ).alias("records_with_rating_count"),
    F.countDistinct(
        F.when(
            F.col("main_accords").isNotNull(),
            F.col("id")
        )
    ).alias("records_with_non_null_accord_field")
)

valid_accord_count = valid_accord_ids.count()
rated_valid_accord_count = (
    valid_accord_ids
    .join(rated_ids, on="id", how="inner")
    .count()
)

valid_accord_without_rating_count = (
    valid_accord_ids
    .join(rated_ids, on="id", how="left_anti")
    .count()
)

profile_row = profile.first().asDict()

profile_row["records_with_valid_accord_field"] = valid_accord_count
profile_row["rated_records_with_valid_accords"] = rated_valid_accord_count
profile_row["valid_accord_records_without_rating"] = valid_accord_without_rating_count

table_i_1 = pd.DataFrame([
    {"Metric": "Total fragrance records", "Value": profile_row["total_records"]},
    {"Metric": "Distinct fragrance IDs", "Value": profile_row["distinct_ids"]},
    {"Metric": "Records with a rating", "Value": profile_row["records_with_rating"]},
    {"Metric": "Records with a rating count", "Value": profile_row["records_with_rating_count"]},
    {"Metric": "Records with a non-null accord field", "Value": profile_row["records_with_non_null_accord_field"]},
    {"Metric": "Records with at least one valid accord", "Value": profile_row["records_with_valid_accord_field"]},
    {"Metric": "Rated records with at least one valid accord", "Value": profile_row["rated_records_with_valid_accords"]},
    {"Metric": "Valid-accord records without a rating", "Value": profile_row["valid_accord_records_without_rating"]}
])

table_i_1["Value"] = table_i_1["Value"].map(lambda x: f"{x:,}")

gt_table_i_1 = make_book_table(
    table_i_1,
    "Dataset Profile",
    "Coverage and completeness of the Fragrantica catalogue"
)
show_book_table(gt_table_i_1)




# ============================================================
# SAVE TABLE I.1 — DATASET PROFILE (PNG)
# ============================================================

import matplotlib.pyplot as plt

output_dir = "/Volumes/main/fragrance_analytics/reference_data/visualizations"
output_path = f"{output_dir}/table_i_1_dataset_profile.png"

fig, ax = plt.subplots(figsize=(10, 4.2))
fig.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])
ax.axis("off")

table = ax.table(
    cellText=table_i_1.values,
    colLabels=table_i_1.columns,
    cellLoc="left",
    colLoc="left",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1, 1.25)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor(BOOK_STYLE["grid"])
    cell.set_linewidth(0.6)
    cell.set_facecolor(BOOK_STYLE["background"])
    cell.set_text_props(
        color=BOOK_STYLE["text"],
        fontfamily=BOOK_TYPOGRAPHY["body"]
    )

    if row == 0:
        cell.set_text_props(
            weight="bold",
            color=BOOK_STYLE["text"]
        )

fig.text(
    0.08, 0.94,
    "Dataset Profile",
    ha="left",
    va="top",
    fontsize=18,
    fontweight="bold",
    color=BOOK_STYLE["text"]
)

fig.text(
    0.08, 0.87,
    "Coverage and completeness of the Fragrantica catalogue",
    ha="left",
    va="top",
    fontsize=11,
    color=BOOK_STYLE["text"]
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.close(fig)

print(f"Saved: {output_path}")


## 4. Figure I.1 — Analytical Framework

A conceptual overview of the research workflow and the five analytical chapters. The framework distinguishes source data, preparation, analytical dimensions and interpretation.

In [0]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path

# OUTPUT DIRECTORY
INTRO_OUTPUT_DIR = Path(
    "/Volumes/main/fragrance_analytics/reference_data/visualizations"
)
INTRO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# FIGURE CANVAS
fig_i_1, ax = plt.subplots(figsize=(15, 8.5), dpi=180)

fig_i_1.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

ax.set_xlim(0, 15)
ax.set_ylim(0, 8.5)
ax.axis("off")


# FRAMEWORK BOXES
def add_framework_box(x, y, width, height, heading, body, accent=False):

    patch = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.025,rounding_size=0.12",
        facecolor=BOOK_STYLE["background"],
        edgecolor=BOOK_STYLE["accent"] if accent else BOOK_STYLE["rule"],
        linewidth=1.5 if accent else 1.0
    )

    ax.add_patch(patch)

    # Box heading
    ax.text(
        x + width / 2,
        y + height * 0.68,
        heading,
        ha="center",
        va="center",
        fontsize=18,
        fontfamily=BOOK_TYPOGRAPHY["body"],
        fontweight="bold",
        color=BOOK_STYLE["text"]
    )

    # Box description
    ax.text(
        x + width / 2,
        y + height * 0.30,
        body,
        ha="center",
        va="center",
        fontsize=15,
        fontfamily=BOOK_TYPOGRAPHY["body"],
        color=BOOK_STYLE["muted"],
        linespacing=1.4
    )


# ARROWS
def add_framework_arrow(start, end):

    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=17,
            linewidth=1.4,
            color=BOOK_STYLE["accent"]
        )
    )


# DATA FLOW: TOP ROW
add_framework_box(
    0.5, 6.15, 4.0, 1.25,
    "DATA SOURCES",
    "Fragrantica fragrance catalogue\nSupplementary sources where applicable"
)

add_framework_box(
    5.5, 6.15, 4.0, 1.25,
    "DATA PREPARATION",
    "Parsing · Cleaning · Normalisation\nValidation · Record-level matching"
)

add_framework_box(
    10.5, 6.15, 4.0, 1.25,
    "ANALYTICAL DATASET",
    "Fragrance records · Accord labels\nRatings · Platform engagement"
)

add_framework_arrow((4.55, 6.77), (5.42, 6.77))
add_framework_arrow((9.55, 6.77), (10.42, 6.77))


# ANALYTICAL CHAPTERS: MIDDLE ROW
chapter_boxes = [
    (0.35, "CHAPTER 1", "Consumer preference\npatterns"),
    (3.3, "CHAPTER 2", "Accord\ncombinations"),
    (6.25, "CHAPTER 3", "Gender and market\nsegmentation"),
    (9.2, "CHAPTER 4", "Brand\npositioning"),
    (12.15, "CHAPTER 5", "Market\nopportunities")
]

for x, chapter, description in chapter_boxes:

    add_framework_box(
        x, 3.55, 2.5, 1.35,
        chapter,
        description,
        accent=True
    )

    # Arrows originate from the bottom edge of the analytical dataset
    add_framework_arrow(
        (12.5, 6.10),
        (x + 1.25, 4.98)
    )


# INTERPRETATION: BOTTOM ROW
add_framework_box(
    3.0, 0.55, 9.0, 1.45,
    "INTERPRETATION AND INDUSTRY IMPLICATIONS",
    "Observed data patterns · Analytical comparisons\n"
    "Potential opportunities and hypotheses",
    accent=True
)

for x, _, _ in chapter_boxes:

    add_framework_arrow(
        (x + 1.25, 3.45),
        (7.5, 2.08)
    )


# FIGURE HEADER
add_book_figure_header(
    fig_i_1,
    "From Fragrance Data to Market Intelligence",
    "Research workflow connecting source data, analytical dimensions "
    "and evidence-based interpretation.",
    section="FRAGRANCE INTELLIGENCE  /  INTRODUCTION"
)

# Adjust header typography for Figure I.1 only
for text_obj in fig_i_1.texts:

    if text_obj.get_text() == "From Fragrance Data to Market Intelligence":
        text_obj.set_fontsize(30)

    elif text_obj.get_text() == "FRAGRANCE INTELLIGENCE  /  INTRODUCTION":
        text_obj.set_fontsize(13)

    elif text_obj.get_text().startswith("Research workflow connecting"):
        text_obj.set_fontsize(16)


# LAYOUT AND EXPORT
fig_i_1.subplots_adjust(
    left=0.025,
    right=0.975,
    top=0.96,
    bottom=0.025
)

fig_i_1.savefig(
    INTRO_OUTPUT_DIR / "figure_i_1_analytical_framework.png",
    dpi=300,
    bbox_inches="tight",
    facecolor=BOOK_STYLE["background"]
)

show_book_matplotlib(fig_i_1)
plt.close(fig_i_1)

print(
    f"Saved: {INTRO_OUTPUT_DIR / 'figure_i_1_analytical_framework.png'}"
)

## 5. Figure I.2 — Dataset Availability Matrix

The matrix displays four mutually exclusive record-level groups based on the availability of a rating and at least one valid accord. It complements the overall dataset profile by making the overlap between the two analysis fields visible.

In [0]:
# ============================================================
# FIGURE I.2 — DATASET AVAILABILITY MATRIX: SOURCE CALCULATIONS
# ============================================================

valid_accord_flag = valid_accord_ids.withColumn(
    "has_valid_accords",
    F.lit(True)
)

availability = (
    df_perfumes
    .select("id", "rating_value")
    .dropDuplicates(["id"])
    .join(valid_accord_flag, on="id", how="left")
    .withColumn("has_rating", F.col("rating_value").isNotNull())
    .withColumn(
        "has_valid_accords",
        F.coalesce(F.col("has_valid_accords"), F.lit(False))
    )
)

availability_counts = availability.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("has_rating") & F.col("has_valid_accords"), 1).otherwise(0)).alias("accord_and_rating"),
    F.sum(F.when(~F.col("has_rating") & F.col("has_valid_accords"), 1).otherwise(0)).alias("accord_only"),
    F.sum(F.when(F.col("has_rating") & ~F.col("has_valid_accords"), 1).otherwise(0)).alias("rating_only"),
    F.sum(F.when(~F.col("has_rating") & ~F.col("has_valid_accords"), 1).otherwise(0)).alias("neither")
).first().asDict()

group_total = (
    availability_counts["accord_and_rating"]
    + availability_counts["accord_only"]
    + availability_counts["rating_only"]
    + availability_counts["neither"]
)

assert group_total == availability_counts["total"], (
    "Availability groups do not reconcile to the total record count."
)

print(f"Availability groups reconciled: {group_total:,} / {availability_counts['total']:,}")

In [0]:
# ============================================================
# FIGURE I.2 — DATASET AVAILABILITY MATRIX
# Ivory editorial treatment with a shared overline/title/subtitle.
# ============================================================

from matplotlib.patches import Rectangle

fig_i_2, ax = plt.subplots(figsize=(12, 7), dpi=180)
fig_i_2.patch.set_facecolor(BOOK_STYLE["background"])
ax.set_facecolor(BOOK_STYLE["background"])

matrix = [
    [availability_counts["accord_and_rating"], availability_counts["accord_only"]],
    [availability_counts["rating_only"], availability_counts["neither"]]
]
total = availability_counts["total"]

row_labels = ["Valid accord data", "No valid accord data"]
col_labels = ["Rating available", "Rating unavailable"]
cell_titles = [
    ["Rated + valid accords", "Valid accords only"],
    ["Rated records only", "Neither available"]
]

for row in range(2):
    for col in range(2):
        value = matrix[row][col]
        pct = value / total * 100
        x, y = col, 1 - row

        ax.add_patch(Rectangle(
            (x, y), 1, 1,
            facecolor=BOOK_STYLE["background"],
            edgecolor=BOOK_STYLE["rule"],
            linewidth=1.1,
            zorder=1
        ))
        ax.add_patch(Rectangle(
            (x + 0.075, y + 0.13), 0.035, 0.74,
            facecolor=BOOK_STYLE["accent"],
            edgecolor="none",
            zorder=2
        ))
        ax.text(
            x + 0.54, y + 0.68, cell_titles[row][col],
            ha="center", va="center",
            fontsize=BOOK_SIZES["category_pt"],
            fontfamily=BOOK_TYPOGRAPHY["body"],
            fontweight="bold",
            color=BOOK_STYLE["accent"]
        )
        ax.text(
            x + 0.54, y + 0.45, f"{value:,}",
            ha="center", va="center",
            fontsize=BOOK_SIZES["matrix_value_pt"],
            fontfamily=BOOK_TYPOGRAPHY["display"],
            fontweight="bold",
            color=BOOK_STYLE["text"]
        )
        ax.text(
            x + 0.54, y + 0.24, f"{pct:.1f}% of catalogue",
            ha="center", va="center",
            fontsize=BOOK_SIZES["value_pt"],
            fontfamily=BOOK_TYPOGRAPHY["body"],
            color=BOOK_STYLE["muted"]
        )

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_xticks([0.5, 1.5], col_labels)
ax.set_yticks([1.5, 0.5], row_labels)
ax.tick_params(length=0, labelsize=BOOK_SIZES["category_pt"], colors=BOOK_STYLE["text"], pad=13)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontfamily(BOOK_TYPOGRAPHY["body"])
    tick.set_fontsize(BOOK_SIZES["category_pt"])
    tick.set_color(BOOK_STYLE["text"])

for spine in ax.spines.values():
    spine.set_visible(False)

add_book_figure_header(
    fig_i_2,
    "Dataset Availability Matrix",
    "Coverage of rating and valid accord data across the full fragrance catalogue.",
    section="FRAGRANCE INTELLIGENCE  /  INTRODUCTION"
)
fig_i_2.text(
    0.5, 0.035,
    f"Total dataset: {total:,} fragrance records. Each quadrant reports count and share of the catalogue.",
    ha="center",
    fontsize=BOOK_SIZES["footer_pt"],
    fontfamily=BOOK_TYPOGRAPHY["body"],
    color=BOOK_STYLE["muted"]
)

plt.subplots_adjust(left=0.16, right=0.96, top=1 - BOOK_FIGURE_CHART_TOP_IN / fig_i_2.get_figheight(), bottom=0.13)
fig_i_2.savefig(
    INTRO_OUTPUT_DIR / "figure_i_2_dataset_availability_matrix.png",
    dpi=300, bbox_inches=None, facecolor=BOOK_STYLE["background"]
)
show_book_matplotlib(fig_i_2)
plt.close(fig_i_2)
print(f"Saved: {INTRO_OUTPUT_DIR / 'figure_i_2_dataset_availability_matrix.png'}")

## 6. Reproducibility notes

- Source table: `main.fragrance_analytics.fragrantica_perfumes`.
- Accord parsing follows the established logic used in Notebook 08.
- A valid accord is a parsed, non-null, non-empty label after lower-case and whitespace normalisation.
- Dataset availability groups are mutually exclusive and are asserted to reconcile to the total number of distinct fragrance records.
- The matrix percentages use the full catalogue as the denominator.
- Output directory: `/Volumes/main/fragrance_analytics/reference_data/visualizations/`.

**Execution note:** run the notebook from top to bottom in Databricks to regenerate the table and figures.